In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
#import warnings
#warnings.filterwarnings("ignore")
import scipy
from scipy import spatial

/home/maltem/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:62: UserWarning: Pandas requires version '1.3.4' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [ ]:
def findindex(alat,alon,plat,plon):
    #finding identical location of pos plat, plon in array alat[],alon[]
    abslat = np.abs(alat-plat)
    abslon = np.abs(alon-plon)
    c = np.maximum(abslon,abslat)
    #latlon_idx = np.argmin(c)
    x, y = np.where(c == np.min(c))
    print(alat[x,y],alon[x,y])
    x=int(x)
    y=int(y)
    return (x,y)

In [2]:
import numpy as np

def findindex(alat, alon, plat, plon):
    # Finding the closest location to (plat, plon) in arrays alat[] and alon[]
    abslat = np.abs(alat - plat)
    abslon = np.abs(alon - plon)
    c = np.maximum(abslon, abslat)
    
    # Find indices of the minimum value in c
    indices = np.where(c == np.min(c))
    
    # Check if multiple indices are returned
    if len(indices[0]) > 1:
        print("Multiple matches found, selecting the first match.")
    
    # Select the first match if there are multiple
    x, y = indices[0][0], indices[1][0]
    
    print(f"Closest latitude: {alat[x, y]}, longitude: {alon[x, y]}")
    return (x, y)


## (not needed anymore) Co-locate Time Axis 

In [ ]:
OMBwave = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/cruise/2024_KVS_deployment.nc')
# Creating Model coordinate
OMBwave.coords["model"] = ("model", ['MF-WAM','MET-ARCWAM','MET-WAM','NRL-COAMPS3','NRL-COAMPS2','NERSC-neXtSIM'])
# "Model simulated 1D"
OMBwave['MODpHs0'] = (['model','trajectory', 'obs_waves_imu'], np.full((len(OMBwave['model']), len(OMBwave['trajectory']), len(OMBwave['obs_waves_imu'])), np.nan))
OMBwave['MODHs0'] = (['model','trajectory', 'obs_waves_imu'], np.full((len(OMBwave['model']), len(OMBwave['trajectory']), len(OMBwave['obs_waves_imu'])), np.nan))
# "Model simulated 2D"
OMBwave['MODelevation_energy_spectrum'] = (['model','trajectory', 'obs_waves_imu', 'frequencies_waves_imu'], np.full((len(OMBwave['model']),len(OMBwave['trajectory']), len(OMBwave['obs_waves_imu']), len(OMBwave['frequencies_waves_imu'])), np.nan))

# "Lat/lon Axis corresponding to wave measurements"
OMBwave['obs_waves_lat'] = (['trajectory', 'obs_waves_imu'], np.full(( len(OMBwave['trajectory']), len(OMBwave['obs_waves_imu'])), np.nan))
OMBwave['obs_waves_lon'] = (['trajectory', 'obs_waves_imu'], np.full(( len(OMBwave['trajectory']), len(OMBwave['obs_waves_imu'])), np.nan))


# Note: we assume time has dimensions (trajectory, obs)
time_coords = pd.to_datetime(OMBwave['time'].values)  # .compute() to get actual values

# Loop through each (trajectory, obs_waves_imu) combination
for trajectory_idx in range(OMBwave.dims['trajectory']):
    for obs_waves_imu_idx in range(OMBwave.dims['obs_waves_imu']):
        
        # Get the corresponding time for the current (trajectory, obs_waves_imu)
        time_waves_imu = pd.to_datetime(OMBwave['time_waves_imu'][trajectory_idx, obs_waves_imu_idx].values)  # .compute() to get the value
        
        # Calculate the time differences (absolute values) between the times
        # Flatten the 2D array of time to a 1D array for comparison
        time_diffs = np.abs(time_coords[trajectory_idx, :] - time_waves_imu)  # Index for trajectory and all obs
        
        # Find the closest time index (within one hour)
        closest_idx = np.argmin(time_diffs)
        
        # Check if the time difference is greater than 1 hour (3600 seconds)
        if time_diffs[closest_idx] <= pd.Timedelta(hours=1):
            # Assign the lat/lon values of the closest time
            OMBwave['obs_waves_lat'].values[trajectory_idx, obs_waves_imu_idx] = OMBwave['lat'].values[trajectory_idx, closest_idx]
            OMBwave['obs_waves_lon'].values[trajectory_idx, obs_waves_imu_idx] = OMBwave['lon'].values[trajectory_idx, closest_idx]
        
        # Otherwise, np.nan will remain as default

# Optionally, you can print out the new variables to check
print(OMBwave['obs_waves_lat'])
print(OMBwave['obs_waves_lon'])


## MF-WAM

In [ ]:
import xarray as xr
import numpy as np

# Load the gridded dataset
modelID = 0 # MF-WAM
ipath = '/lustre/storeB/users/maltem/SALIENSEAS/SvalMIZ2024/models/'
modelSWH = xr.open_mfdataset(ipath + 'MF-WAM/swh_da.nc')

# Prepare OMBwave trajectory data
trajectory_lat = OMBwave['obs_waves_lat']
trajectory_lon = OMBwave['obs_waves_lon']
trajectory_time = OMBwave['time_waves_imu']

# Initialize an empty list to store colocated values
colocated_values = []

# Loop through trajectories
for traj_idx in range(OMBwave.dims['trajectory']):
    # Extract latitude, longitude, and time for this trajectory
    lat_values = trajectory_lat.isel(trajectory=traj_idx)
    lon_values = trajectory_lon.isel(trajectory=traj_idx)
    time_values= trajectory_time.isel(trajectory=traj_idx)
    # Mask invalid (NaN) coordinates
    valid_mask = ~np.isnan(lat_values) & ~np.isnan(lon_values)
    
    # Use the valid mask to filter lat, lon, and their corresponding time values
    valid_lat = lat_values.where(valid_mask, drop=True)
    valid_lon = lon_values.where(valid_mask, drop=True)
    valid_time = time_values.where(valid_mask, drop=True)

    # Use xarray's `sel` for colocation
    for t_idx in range(len(valid_time)):
        colocated = modelSWH['var100'].sel(
            time=valid_time[t_idx].values,
            lat=valid_lat[t_idx].values,
            lon=valid_lon[t_idx].values,
            method='nearest'
        ).values

        
        OMBwave.MODHs0[modelID,traj_idx,t_idx]=colocated

OMBwave.to_netcdf('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM.nc')       

## MET-ARCWAM

In [ ]:
OMBwave = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM.nc')

import xarray as xr
import numpy as np
import cartopy.crs as ccrs

# Load the gridded dataset
modelID = 1 # MET-ARCWAM
modelSWH = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/WaveDataAnimation/Out.nc')

# Extract the rotated latitude and longitude
rlat = modelSWH.rlat.values
rlon = modelSWH.rlon.values 

# Create a meshgrid for rlat and rlon
rlat_grid, rlon_grid = np.meshgrid(rlon, rlat)

# Create the rotated latitude/longitude projection using the PROJ string from metadata
rotated_pole_90 = ccrs.RotatedPole(pole_longitude=140, pole_latitude=25)
rotated_pole_proj = pyproj.CRS.from_proj4(str(rotated_pole_90))
# Define the geographic lat/lon projection (WGS84)
geo_proj = pyproj.CRS("EPSG:4326")  # WGS84 (latitude, longitude)

# Create a transformer object to convert from rotated to lat/lon
transformer = pyproj.Transformer.from_crs(rotated_pole_proj, geo_proj, always_xy=True)

# Perform the transformation (inverse transformation)
lon, lat = transformer.transform(rlat_grid, rlon_grid)

# Prepare OMBwave trajectory data
trajectory_lat = OMBwave['obs_waves_lat']
trajectory_lon = OMBwave['obs_waves_lon']
trajectory_time = OMBwave['time_waves_imu']

# Initialize an empty list to store colocated values
colocated_values = []

# Loop through trajectories
for traj_idx in range(OMBwave.dims['trajectory']):
    
    
# Extract latitude, longitude, and time for this trajectory
    lat_values = trajectory_lat.isel(trajectory=traj_idx)
    lon_values = trajectory_lon.isel(trajectory=traj_idx)
    time_values= trajectory_time.isel(trajectory=traj_idx)
    # Mask invalid (NaN) coordinates
    valid_mask = ~np.isnan(lat_values) & ~np.isnan(lon_values)
    # Compute the valid mask if it's a Dask array
    valid_mask = valid_mask.compute() if isinstance(valid_mask, xr.DataArray) else valid_mask
    # Use the valid mask to filter lat, lon, and their corresponding time values
    valid_lat = lat_values.where(valid_mask, drop=True)
    valid_lon = lon_values.where(valid_mask, drop=True)
    valid_time = time_values.where(valid_mask, drop=True)

    # Use xarray's `sel` for colocation
    
       
    for t_idx in range(len(valid_time)):
        
        time_t = valid_time[t_idx].values          # Trajectory time for this step
        model_times = modelSWH['time'].values     # Model time values (ensure it's in the same units)
        time_diff = np.abs(model_times - time_t)  # Absolute time difference

        # Find the index of the closest model time
        closest_time_idx = np.argmin(time_diff)
        closest_swh = modelSWH['hs'].isel(time=closest_time_idx).values
        
        [ix,jx] = findindex(lat,lon,valid_lat[t_idx].values,valid_lon[t_idx].values)
        
        colocated = modelSWH['hs'][closest_time_idx,ix,jx].values
        #print(lat[ix,jx],valid_lat[t_idx].values)
        

        
        OMBwave.MODHs0[modelID,traj_idx,t_idx]=colocated

modelSWH.close()    
OMBwave.to_netcdf('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM.nc')  

## NRL COAMPS

In [ ]:
OMBwave = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM_NRL-COAMPS3_nd.nc')

import xarray as xr
import numpy as np
import cartopy.crs as ccrs

# Load the gridded dataset
modelID = 4 # NRL-COAMPS2
ipath = '/lustre/storeB/users/maltem/SALIENSEAS/SvalMIZ2024/models/'

modelSWH = xr.open_dataset(ipath + 'NRL-COAMPS2/COAMPS2/ww3.202404_ef.nc', decode_coords="all")
# Assign coordinates for latitude and longitude
latitude = modelSWH['latitude'][:, 0].values  # Extract unique latitudes (assume same across longitude)
longitude = modelSWH['longitude'][0, :].values  # Extract unique longitudes (assume same across latitude)

lat = modelSWH['latitude'].values
lon = modelSWH['longitude'].values

# Assign coordinates for wave frequency (f) and time
f = modelSWH['f'].values
time = modelSWH['time'].values
# Update the dataset with proper coordinates
modelSWH = modelSWH.assign_coords(
    latitude=("latitude", latitude),
    longitude=("longitude", longitude),
    f=("f", f),
    time=("time", time)
)

# Drop the old latitude and longitude variables (optional)
modelSWH = modelSWH.drop_vars(["latitude", "longitude"])

# Prepare OMBwave trajectory data
trajectory_lat = OMBwave['obs_waves_lat']
trajectory_lon = OMBwave['obs_waves_lon']
trajectory_time = OMBwave['time_waves_imu']      

                            
# Initialize an empty list to store colocated values
colocated_values = []
frequencies = modelSWH['f'].values  # Frequency bins
delta_f = np.diff(frequencies, prepend=frequencies[0])

# Loop through trajectories
for traj_idx in range(OMBwave.dims['trajectory']):
    
# Extract latitude, longitude, and time for this trajectory
    lat_values = trajectory_lat.isel(trajectory=traj_idx)
    lon_values = trajectory_lon.isel(trajectory=traj_idx)
    time_values= trajectory_time.isel(trajectory=traj_idx)
    # Mask invalid (NaN) coordinates
    valid_mask = ~np.isnan(lat_values) & ~np.isnan(lon_values)
    # Compute the valid mask if it's a Dask array
    valid_mask = valid_mask.compute() if isinstance(valid_mask, xr.DataArray) else valid_mask
    # Use the valid mask to filter lat, lon, and their corresponding time values
    valid_lat = lat_values.where(valid_mask, drop=True)
    valid_lon = lon_values.where(valid_mask, drop=True)
    valid_time = time_values.where(valid_mask, drop=True)

    # Use xarray's `sel` for colocation
    
        
    for t_idx in range(len(valid_time)):
        
        time_t = valid_time[t_idx].values          # Trajectory time for this step
        model_times = modelSWH['time'].values     # Model time values (ensure it's in the same units)
        time_diff = np.abs(model_times - time_t)  # Absolute time difference

        
        # Only proceed if the minimum time difference is less than 1 hour (3600 seconds)
        if np.min(time_diff) <= np.timedelta64(1, 'h'):
            closest_time_idx = np.argmin(time_diff)
            closest_ef = modelSWH['ef'].isel(time=closest_time_idx).values
            
            # Find the spatial indices of the closest grid point
            [ix, jx] = findindex(lat, lon, valid_lat[t_idx].values, valid_lon[t_idx].values)
            
            ef_scaled = closest_ef[:, ix, jx] 
            S_f = 10 ** ef_scaled
            
            # Compute m0 (zeroth moment of the spectrum)
            m0 = (S_f * delta_f).sum()
            Hs = 4 * np.sqrt(m0)
            print (time_t,traj_idx,Hs)
            # Save the colocated significant wave height
            OMBwave.MODHs0[modelID, traj_idx, t_idx] = Hs
        else:
            # Assign NaN if no colocated value is within 1 hour
            OMBwave.MODHs0[modelID, traj_idx, t_idx] = np.nan
        

modelSWH.close()    
OMBwave.to_netcdf('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM_NRL-COAMPS3.nc')  

                            

In [ ]:
OMBwave.to_netcdf('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM_NRL-COAMPS23.nc')  


## NRL NextSIM-WW3

In [3]:
OMBwave = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM_NRL-COAMPS23.nc')
modelID = 5
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
ipath = '/lustre/storeB/users/maltem/SALIENSEAS/SvalMIZ2024/models/'
modelSWH = xr.open_dataset(ipath + 'NERSC-NextSIM-WW3/april2024_ww3nx.12x12.MetNo/analysis/ww3_outputs/2024/ww3.202405_hs.nc', decode_coords="all")
lat = modelSWH['latitude']
lon = modelSWH['longitude']

# Prepare OMBwave trajectory data
trajectory_lat = OMBwave['obs_waves_lat']
trajectory_lon = OMBwave['obs_waves_lon']
trajectory_time = OMBwave['time_waves_imu']                           

# Initialize an empty list to store colocated values
colocated_values = []

# Loop through trajectories
for traj_idx in range(OMBwave.dims['trajectory']):
    
# Extract latitude, longitude, and time for this trajectory
    lat_values = trajectory_lat.isel(trajectory=traj_idx)
    lon_values = trajectory_lon.isel(trajectory=traj_idx)
    time_values= trajectory_time.isel(trajectory=traj_idx)
    # Mask invalid (NaN) coordinates
    valid_mask = ~np.isnan(lat_values) & ~np.isnan(lon_values)
    # Compute the valid mask if it's a Dask array
    valid_mask = valid_mask.compute() if isinstance(valid_mask, xr.DataArray) else valid_mask
    # Use the valid mask to filter lat, lon, and their corresponding time values
    valid_lat = lat_values.where(valid_mask, drop=True)
    valid_lon = lon_values.where(valid_mask, drop=True)
    valid_time = time_values.where(valid_mask, drop=True)

    # Use xarray's `sel` for colocation
    
        
    for t_idx in range(len(valid_time)):
        
        time_t = valid_time[t_idx].values          # Trajectory time for this step
        model_times = modelSWH['time'].values     # Model time values (ensure it's in the same units)
        time_diff = np.abs(model_times - time_t)  # Absolute time difference

        
        # Only proceed if the minimum time difference is less than 1 hour (3600 seconds)
        if np.min(time_diff) <= np.timedelta64(1, 'h'):
            closest_time_idx = np.argmin(time_diff)
            closest_hs = modelSWH['hs'].isel(time=closest_time_idx).values
            
            # Find the spatial indices of the closest grid point
            [ix, jx] = findindex(np.array(lat), np.array(lon), valid_lat[t_idx].values, valid_lon[t_idx].values)
            print(modelID,traj_idx,t_idx,closest_hs[ix,jx],np.isnan(closest_hs[ix, jx]))
            # Save the colocated significant wave height
            OMBwave.MODHs0[modelID, traj_idx, t_idx] = 0.1#closest_hs[ix,jx]
           
            
        else:
            # Assign NaN if no colocated value is within 1 hour
            OMBwave.MODHs0[modelID, traj_idx, t_idx] = 0.1#np.nan
        

modelSWH.close()    
OMBwave.to_netcdf('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM_NRL-COAMPS3_nextsim2.nc')  


Closest latitude: 80.4469985961914, longitude: 17.354000091552734
5 2 246 0.02975731 False
Closest latitude: 80.60099792480469, longitude: 17.649999618530273
5 2 258 0.030790184 False
Closest latitude: 80.60099792480469, longitude: 17.649999618530273
5 2 259 0.02934886 False
Closest latitude: 80.60099792480469, longitude: 17.649999618530273
5 2 261 0.017674288 False
Closest latitude: 80.4469985961914, longitude: 17.354000091552734
5 2 263 0.12372441 False
Closest latitude: 80.7030029296875, longitude: 17.323999404907227
5 2 264 0.0018914294 False
Closest latitude: 80.7030029296875, longitude: 17.323999404907227
5 2 266 0.0017977481 False
Closest latitude: 80.7030029296875, longitude: 17.323999404907227
5 2 267 0.0020052076 False
Closest latitude: 80.4469985961914, longitude: 17.354000091552734
5 2 269 0.20492505 False
Closest latitude: 80.4469985961914, longitude: 17.354000091552734
5 2 270 0.18003501 False
Closest latitude: 80.4469985961914, longitude: 17.354000091552734
5 2 272 0.151

5 2 399 0.012323771 False
Closest latitude: 81.26300048828125, longitude: 16.238000869750977
5 2 401 0.009285024 False
Closest latitude: 81.26300048828125, longitude: 16.238000869750977
5 2 402 0.008160076 False
Closest latitude: 81.10700225830078, longitude: 15.944999694824219
5 2 404 0.009493637 False
Closest latitude: 81.10700225830078, longitude: 15.944999694824219
5 2 405 0.0063909017 False
Closest latitude: 81.00599670410156, longitude: 16.301000595092773
5 2 407 0.013485813 False
Closest latitude: 81.00599670410156, longitude: 16.301000595092773
5 2 408 0.008435106 False
Closest latitude: 80.8499984741211, longitude: 16.014999389648438
5 2 410 0.052600414 False
Closest latitude: 80.8499984741211, longitude: 16.014999389648438
5 2 411 0.028614286 False
Closest latitude: 80.75, longitude: 16.361000061035156
5 2 413 0.13314322 False
Closest latitude: 80.90499877929688, longitude: 16.649999618530273
5 2 414 0.024441449 False
Closest latitude: 80.90499877929688, longitude: 16.6499996

Closest latitude: 81.01200103759766, longitude: 17.941999435424805
5 4 308 0.0020020895 False
Closest latitude: 81.01200103759766, longitude: 17.941999435424805
5 4 309 0.0026964734 False
Closest latitude: 81.01200103759766, longitude: 17.941999435424805
5 4 311 0.004452412 False
Closest latitude: 81.01200103759766, longitude: 17.941999435424805
5 4 312 0.0068759974 False
Closest latitude: 81.01200103759766, longitude: 17.941999435424805
5 4 314 0.010005838 False
Closest latitude: 81.26899719238281, longitude: 17.927000045776367
5 4 315 0.004180368 False
Closest latitude: 81.26899719238281, longitude: 17.927000045776367
5 4 317 0.0055190446 False
Closest latitude: 81.26899719238281, longitude: 17.927000045776367
5 4 318 0.007204372 False
Closest latitude: 81.11399841308594, longitude: 17.604000091552734
5 4 320 0.013367076 False
Closest latitude: 81.11399841308594, longitude: 17.604000091552734
5 4 321 0.015642006 False
Closest latitude: 81.11399841308594, longitude: 17.604000091552734

Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 447 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 448 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 450 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 451 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 453 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 454 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 456 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 457 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 459 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 460 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 462 nan True
Closest latitude: 80.14099884033203, longitude: 18.28499984741211
5 4 463 nan True
Clos

Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 57 0.513538 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 58 0.5395281 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 59 0.5352943 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 61 0.42395258 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 62 0.53841937 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 64 0.4818332 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 65 0.2914743 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 67 0.28177193 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 68 0.08536826 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 70 0.07433193 False
Closest latitude: 80.08899688720703, longitude: 17.690000534057617
5 5 71 0.25931466 False
Close

...

Closest latitude: 80.6989974975586, longitude: 20.496999740600586
5 26 88 0.0038046995 False
Closest latitude: 80.6989974975586, longitude: 20.496999740600586
5 26 90 0.003857652 False
Closest latitude: 80.6989974975586, longitude: 20.496999740600586
5 26 91 0.0068620383 False
Closest latitude: 80.6989974975586, longitude: 20.496999740600586
5 26 93 0.0068596955 False
Closest latitude: 80.8030014038086, longitude: 20.198999404907227
5 26 94 0.0028909275 False
Closest latitude: 80.6510009765625, longitude: 19.854999542236328
5 26 96 0.008388344 False
Closest latitude: 80.6510009765625, longitude: 19.854999542236328
5 26 97 0.009469391 False
Closest latitude: 80.6510009765625, longitude: 19.854999542236328
5 26 99 0.008265264 False
Closest latitude: 80.6510009765625, longitude: 19.854999542236328
5 26 100 0.006453819 False
Closest latitude: 80.6510009765625, longitude: 19.854999542236328
5 26 102 0.0065690866 False
Closest latitude: 80.49800109863281, longitude: 19.520999908447266
5 26 1

Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 337 0.004610604 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 338 0.0040617418 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 340 0.003544637 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 341 0.0033075656 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 343 0.003071675 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 344 0.0028329587 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 346 0.0026258167 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 347 0.0034058043 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 349 0.003512203 False
Closest latitude: 81.31999969482422, longitude: 18.604000091552734
5 27 350 0.0048039807 False
Closest latitude: 81.21700286865234, longitude: 18.938

Closest latitude: 81.47200012207031, longitude: 17.222999572753906
5 28 264 0.0009733737 False
Closest latitude: 81.31700134277344, longitude: 16.905000686645508
5 28 266 0.0025932163 False
Closest latitude: 81.31700134277344, longitude: 16.905000686645508
5 28 267 0.0025861005 False
Closest latitude: 81.31700134277344, longitude: 16.905000686645508
5 28 269 0.0024474163 False
Closest latitude: 81.31700134277344, longitude: 16.905000686645508
5 28 270 0.0049720462 False
Closest latitude: 81.47200012207031, longitude: 17.222999572753906
5 28 272 0.00087193365 False
Closest latitude: 81.47200012207031, longitude: 17.222999572753906
5 28 273 0.00069953833 False
Closest latitude: 81.47200012207031, longitude: 17.222999572753906
5 28 275 0.0006286831 False
Closest latitude: 81.21600341796875, longitude: 17.257999420166016
5 28 276 0.011380282 False
Closest latitude: 81.47200012207031, longitude: 17.222999572753906
5 28 278 0.0005383625 False
Closest latitude: 81.47200012207031, longitude: 1

Closest latitude: 80.80400085449219, longitude: 16.989999771118164
5 32 231 0.0016206391 False
Closest latitude: 80.80400085449219, longitude: 16.989999771118164
5 32 232 0.0015539281 False
Closest latitude: 80.80400085449219, longitude: 16.989999771118164
5 32 234 0.0016541007 False
Closest latitude: 80.80400085449219, longitude: 16.989999771118164
5 32 235 0.0015358989 False
Closest latitude: 80.80400085449219, longitude: 16.989999771118164
5 32 237 0.0015411928 False
Closest latitude: 80.80400085449219, longitude: 16.989999771118164
5 32 238 0.0010529475 False
Closest latitude: 80.80400085449219, longitude: 16.989999771118164
5 32 240 0.0010644351 False
Closest latitude: 80.64900207519531, longitude: 16.698999404907227
5 32 241 0.0025812252 False
Closest latitude: 80.64900207519531, longitude: 16.698999404907227
5 32 243 0.0025348843 False
Closest latitude: 80.64900207519531, longitude: 16.698999404907227
5 32 244 0.002365817 False
Closest latitude: 80.64900207519531, longitude: 16.

RecursionError: maximum recursion depth exceeded while calling a Python object

In [ ]:
OMBwave['MODHs0'][5,32,346].values

In [ ]:
import sys
sys.setrecursionlimit(5000)
OMBwave.to_netcdf('/home/maltem/test.nc')  


In [ ]:
OMBwave = xr.open_mfdataset('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM_NRL-COAMPS23.nc')
OMBwave.MODHs0[5, 33, 1] = 0.1
OMBwave.to_netcdf('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_MF-WAM_MET-WAM_NRL-COAMPS3_new2.nc')  


In [ ]:
OMBwave = OMBwave.load()  # Load all data into memory
OMBwave.to_netcdf('/home/maltem/work/python/data/SvalMIZ2024/colocatedFiles/dataset_wave_all.nc')

In [ ]:
np.array(lon)

In [ ]:
ef_scaled = modelSWH.ef[0,:,188,53]
S_f = 10 ** ef_scaled
m0 = (S_f * delta_f).sum()
Hs = 4 * np.sqrt(m0)
print(Hs)
plt.plot(frequencies,S_f)

In [ ]:
OMBwave